# Logdet Method Profiling Across Matrix Sizes

This notebook profiles the runtime of different log-determinant strategies used in bayespecon:

- `exact`: symbolic determinant route
- `eigenvalue`: spectral precompute route
- `grid`: spline interpolation over a precomputed grid
- `full`: sparse-LU grid (MATLAB-style `lndetfull`)
- `int`: sparse-LU + spline interpolation (MATLAB-style `lndetint`)
- `mc`: Monte Carlo trace approximation (MATLAB-style `lndetmc`)
- `ichol`: ILU-based approximation (MATLAB-style `lndetichol` analog)

For each matrix size, we report:

- **setup time**: build + compile callable logdet function
- **evaluation time**: average cost to evaluate at many rho values

In [ ]:
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytensor
import pytensor.tensor as pt

from bayespecon.logdet import make_logdet_fn

In [ ]:
def make_line_w(n: int) -> np.ndarray:
    """Create a row-standardized line-lattice W matrix."""
    W = np.zeros((n, n), dtype=np.float64)
    idx = np.arange(n)
    W[idx[1:], idx[:-1]] = 1.0
    W[idx[:-1], idx[1:]] = 1.0
    row_sums = W.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0.0] = 1.0
    return W / row_sums


def compile_logdet_callable(W: np.ndarray, method: str):
    """Return a compiled callable f(rho) and its setup time in seconds."""
    t0 = time.perf_counter()
    rho = pt.scalar('rho')
    expr = make_logdet_fn(W, method=method, rho_min=-0.95, rho_max=0.95)(rho)
    fn = pytensor.function([rho], expr)
    setup_s = time.perf_counter() - t0
    return fn, setup_s


def bench_eval_seconds(fn, rhos: np.ndarray, repeats: int = 5) -> float:
    """Median per-call evaluation latency in microseconds."""
    run_times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        for r in rhos:
            _ = fn(float(r))
        elapsed = time.perf_counter() - t0
        run_times.append(elapsed / len(rhos))
    return float(np.median(run_times))

In [ ]:
@dataclass
class ProfileConfig:
    # Include larger sizes to expose high-n behavior.
    sizes: tuple[int, ...] = (40, 80, 120, 180, 260, 360, 520, 720, 1000, 2000, 5000, 10000)
    methods: tuple[str, ...] = ('exact', 'eigenvalue', 'grid', 'full', 'int', 'mc', 'ichol')
    method_max_n: dict = None
    eval_points: int = 80
    eval_repeats: int = 3

    def __post_init__(self):
        if self.method_max_n is None:
            # Practical safety caps for this dense benchmark setup.
            # - exact grows quickly, keep only small n
            # - grid allocates an n_grid x n x n tensor internally
            # - eigenvalue requires dense eigendecomposition
            # - full/int/ichol use sparse factorizations across rho grid
            # - mc uses stochastic trace approximation
            self.method_max_n = {
                'exact': 120,
                'eigenvalue': 2000,
                'grid': 1000,
                'full': 1000,
                'int': 5000,
                'mc': 5000,
                'ichol': 5000,
            }


cfg = ProfileConfig()
rho_grid = np.linspace(-0.9, 0.9, cfg.eval_points)
results = []
skipped = []

for n in cfg.sizes:
    W = make_line_w(n)
    print(f'Profiling n={n}...')
    for method in cfg.methods:
        if n > cfg.method_max_n[method]:
            skipped.append({'n': n, 'method': method, 'reason': 'above method_max_n cap'})
            continue
        try:
            fn, setup_s = compile_logdet_callable(W, method)
            eval_s = bench_eval_seconds(fn, rho_grid, repeats=cfg.eval_repeats)
            results.append({
                'n': n,
                'method': method,
                'setup_ms': 1e3 * setup_s,
                'eval_us': 1e6 * eval_s,
            })
        except Exception as exc:
            skipped.append({'n': n, 'method': method, 'reason': f'failed: {type(exc).__name__}: {exc}'})

res = pd.DataFrame(results).sort_values(['method', 'n']).reset_index(drop=True)
if not res.empty:
    # Total cost for a full rho sweep used in this notebook.
    res['total_ms'] = res['setup_ms'] + (res['eval_us'] * cfg.eval_points / 1e3)
res

In [ ]:
if res.empty:
    raise RuntimeError('No profiling results were generated.')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

for method, grp in res.groupby('method'):
    grp = grp.sort_values('n')
    axes[0].plot(grp['n'], grp['setup_ms'], marker='o', label=method)
    axes[1].plot(grp['n'], grp['eval_us'], marker='o', label=method)
    axes[2].plot(grp['n'], grp['total_ms'], marker='o', label=method)

axes[0].set_title('Setup Time vs n')
axes[0].set_xlabel('matrix size n')
axes[0].set_ylabel('setup time (ms)')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Evaluation Time vs n')
axes[1].set_xlabel('matrix size n')
axes[1].set_ylabel('time per rho eval (us)')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

axes[2].set_title(f'Total Time vs n (setup + {cfg.eval_points} evals)')
axes[2].set_xlabel('matrix size n')
axes[2].set_ylabel('total time (ms)')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.legend()

plt.show()

In [ ]:
summary = (
    res.pivot_table(index='n', columns='method', values=['setup_ms', 'eval_us', 'total_ms'])
       .sort_index()
)
display(summary)

if skipped:
    skipped_df = pd.DataFrame(skipped).sort_values(['n', 'method']).reset_index(drop=True)
    print('Skipped combinations (due to safety caps or failures):')
    display(skipped_df)

## Coefficient and Fit-Time Comparison Across Logdet Methods

This section estimates the same SAR model using each `logdet_method` and compares:

- posterior mean coefficients (`rho`, `beta_0`, `beta_1`, `beta_2`)
- total wall-clock time to estimate each model

To keep this section runnable in docs contexts, sampling is intentionally modest.

In [ ]:
import pymc as pm
from bayespecon import dgp


def simulate_sar_data(n: int = 80, seed: int = 2026):
    """Simulate a small SAR dataset for method comparison."""
    rng = np.random.default_rng(seed)
    W = make_line_w(n)

    beta_true = np.array([1.0, 0.8, -0.5], dtype=np.float64)
    rho_true = 0.35
    sigma_true = 0.7

    sim = dgp.simulate_sar(
        W=W,
        rho=rho_true,
        beta=beta_true,
        sigma=sigma_true,
        rng=rng,
    )
    return sim["y"], sim["X"], sim["W_dense"]


def fit_sar_for_method(y, X, W, method: str, draws: int = 120, tune: int = 120, seed: int = 2026):
    """Fit SAR with a specific logdet method and return posterior means + runtime."""
    t0 = time.perf_counter()

    with pm.Model() as model:
        rho = pm.Uniform("rho", lower=-0.95, upper=0.95)
        beta = pm.Normal("beta", mu=0.0, sigma=2.0, shape=X.shape[1])
        sigma = pm.HalfNormal("sigma", sigma=1.0)

        mu = rho * (W @ y) + pt.dot(X, beta)
        pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

        pm.Potential(
            "jacobian",
            make_logdet_fn(W, method=method, rho_min=-0.95, rho_max=0.95)(rho),
        )

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=2,
            cores=1,
            random_seed=seed,
            progressbar=False,
            compute_convergence_checks=False,
        )

    elapsed_s = time.perf_counter() - t0
    beta_mean = idata.posterior["beta"].mean(("chain", "draw")).to_numpy()
    rho_mean = float(idata.posterior["rho"].mean(("chain", "draw")).to_numpy())

    return {
        "method": method,
        "total_time_s": elapsed_s,
        "rho": rho_mean,
        "beta_0": float(beta_mean[0]),
        "beta_1": float(beta_mean[1]),
        "beta_2": float(beta_mean[2]),
    }


methods_for_model = ["exact", "eigenvalue", "grid", "full", "int", "mc", "ichol"]
y_model, X_model, W_model = simulate_sar_data(n=80)

model_rows = []
for m in methods_for_model:
    print(f"Estimating SAR with method={m}...")
    try:
        model_rows.append(fit_sar_for_method(y_model, X_model, W_model, method=m))
    except Exception as exc:
        model_rows.append(
            {
                "method": m,
                "total_time_s": np.nan,
                "rho": np.nan,
                "beta_0": np.nan,
                "beta_1": np.nan,
                "beta_2": np.nan,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

coef_compare = pd.DataFrame(model_rows)
coef_compare = coef_compare.sort_values("total_time_s", na_position="last").reset_index(drop=True)

if "eigenvalue" in coef_compare["method"].values:
    base = coef_compare.loc[coef_compare["method"] == "eigenvalue", ["rho", "beta_0", "beta_1", "beta_2"]].iloc[0]
    for col in ["rho", "beta_0", "beta_1", "beta_2"]:
        coef_compare[f"delta_vs_eigen_{col}"] = coef_compare[col] - base[col]

display(coef_compare)

## Notes

- `exact` is expected to scale poorly and is intentionally capped at smaller n in this notebook.
- `eigenvalue` usually has moderate setup cost and very fast repeated evaluations.
- `grid` has an upfront grid-construction cost but can be competitive for repeated evaluation scenarios.
- `full` and `int` are sparse-LU variants inspired by the MATLAB toolbox routines.
- `mc` is stochastic and may vary slightly run-to-run unless random seeds are fixed.
- `ichol` uses an ILU approximation that can trade precision for speed depending on sparsity.
- Adjust `ProfileConfig.sizes` and `method_max_n` for deeper stress tests.